In [0]:
CREATE TABLE IF NOT EXISTS adb_rtp.gold.reporting_dim_product_gold_SCDTYPE1
 LIKE adb_rtp.gold.reporting_dim_product_gold
--Create a like table of the source table for which SCD is needed

In [0]:
SELECT * FROM adb_rtp.gold.reporting_dim_product_gold_SCDTYPE1
WHERE PRODUCT_NAME='Onion'

In [0]:
UPDATE adb_rtp.silver.daily_pricing_silver
SET PRODUCTGROUP_NAME='Ground Vegetable',
lakehouse_updated_date = current_timestamp()
WHERE PRODUCT_NAME='Onion'

In [0]:
CREATE OR REPLACE TABLE adb_rtp.silver.reporting_dim_product_stage_1 AS
SELECT 
 DISTINCT PRODUCT_NAME
 ,PRODUCTGROUP_NAME
FROM adb_rtp.silver.daily_pricing_silver
WHERE lakehouse_updated_date > (SELECT nvl(max(PROCESSED_TABLE_DATETIME),'1900-01-01') FROM adb_rtp.processrunlogs.DELTALAKEHOUSE_PROCESS_RUNS 
WHERE process_name = 'reportingDimensionTablesLoadScdType1' AND process_status = 'Completed' );

In [0]:
SELECT DISTINCT product_name, PRODUCTGROUP_NAME, product_id FROM adb_rtp.silver.daily_pricing_silver

In [0]:
--CREATE OR REPLACE TABLE adb_rtp.silver.reporting_dim_product_stage_2 AS 
SELECT 
  silverDim.PRODUCT_NAME
  ,silverDim.PRODUCTGROUP_NAME
  ,goldDim.PRODUCT_NAME AS GOLD_PRODUCT_NAME
 ,CASE WHEN goldDim.PRODUCT_NAME IS NULL
 THEN ROW_NUMBER() OVER (  ORDER BY silverDim.PRODUCT_NAME,silverDim.PRODUCTGROUP_NAME) 
 ELSE goldDim.PRODUCT_ID END as PRODUCT_ID
 ,current_timestamp() as lakehouse_inserted_date
 ,current_timestamp() as lakehouse_updated_date
FROM adb_rtp.silver.reporting_dim_product_stage_1 silverDim
LEFT OUTER JOIN adb_rtp.gold.reporting_dim_product_gold_SCDTYPE1 goldDim
ON silverDim.PRODUCT_NAME= goldDim.PRODUCT_NAME
WHERE goldDim.PRODUCT_NAME IS NULL OR silverDim.PRODUCTGROUP_NAME <> goldDim.PRODUCTGROUP_NAME

--Need to ensure that we are pulling the right product_id when handling SCD since we are not inluding new recors for SCD, we should make sure to pull in the existing product_id using CASE statement

In [0]:
SELECT * FROM adb_rtp.silver.reporting_dim_product_stage_2

In [0]:
CREATE OR REPLACE TABLE adb_rtp.silver.reporting_dim_product_stage_3 AS 
SELECT
  silverDim.PRODUCTGROUP_NAME
  ,silverDim.PRODUCT_NAME
,CASE WHEN GOLD_PRODUCT_NAME IS NULL
THEN 
silverDim.PRODUCT_ID + PREV_MAX_SK_ID 
ELSE PRODUCT_ID END as PRODUCT_ID
,current_timestamp() as lakehouse_inserted_date
,current_timestamp() as lakehouse_updated_date
FROM 
adb_rtp.silver.reporting_dim_product_stage_2 silverDim
CROSS JOIN (SELECT nvl(MAX(PRODUCT_ID),0) as PREV_MAX_SK_ID FROM adb_rtp.gold.reporting_dim_product_gold_SCDTYPE1 ) goldDim;


In [0]:
SELECT * FROM adb_rtp.silver.reporting_dim_product_stage_3

In [0]:
MERGE INTO adb_rtp.gold.reporting_dim_product_gold_SCDTYPE1 goldDim
USING adb_rtp.silver.reporting_dim_product_stage_3 silverDim
ON silverDim.PRODUCT_NAME = goldDim.PRODUCT_NAME
WHEN MATCHED THEN
UPDATE SET goldDim.PRODUCTGROUP_NAME=silverDim.PRODUCTGROUP_NAME
           ,goldDim.lakehouse_updated_date=current_timestamp()
WHEN NOT MATCHED THEN
INSERT *

--Merge can handle insert, update and delete in one single statement. In this case we are handling both the records which needs to be updated when there is a change in the product_name column as well as include the records which are new.

In [0]:
SELECT * FROM adb_rtp.gold.reporting_dim_product_gold_SCDTYPE1
WHERE PRODUCT_NAME='Onion'

In [0]:
INSERT INTO  adb_rtp.processrunlogs.DELTALAKEHOUSE_PROCESS_RUNS(PROCESS_NAME,PROCESSED_TABLE_DATETIME,PROCESS_STATUS)
SELECT 'reportingDimensionTablesLoadScdType1' , max(lakehouse_updated_date) ,'Completed' FROM adb_rtp.silver.daily_pricing_silver